# Synthetic data generation for testing causal inference methods

Synthetic data based on:

* Baseline mortality is 10%
* Patients are aged 60-95, uniformally distrubuted. Use integers.
* Mortality increases by 1 percentage point every year above age 60
* Ethnicity is 70% white, 20% asian, 10% black
* Mortality increases 5% points for asian patients and 10% for black patients
* Patients are split 50:50 male:female - have 0/1 for male
* Mortality increases 10% points for males
* Baseline thrombolysis treatment is 20%
* Thrombolysis rate is reduced 5% for asian ptients and 10% for black patients
* Thrombolysis rate is halved if aged more than 80
* The baseline effect of use of thrombolysis is to halve mortality
* Thrombolysis is half as effective in asian and black people
* Thrombolysis is half as effective if aged 80%

Code generated by Claude 4.6

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
N = 25000

# --- Demographics ---
age = np.random.randint(60, 96, size=N)  # 60–95 inclusive
ethnicity = np.random.choice(['white', 'asian', 'black'], size=N, p=[0.70, 0.20, 0.10])
male = np.random.randint(0, 2, size=N)  # 0=female, 1=male

# --- Thrombolysis probability ---
thrombolysis_prob = np.full(N, 0.20)
thrombolysis_prob[ethnicity == 'asian'] -= 0.05
thrombolysis_prob[ethnicity == 'black'] -= 0.10
thrombolysis_prob[age > 80] /= 2
thrombolysis_prob = np.clip(thrombolysis_prob, 0, 1)

thrombolysis = np.random.binomial(1, thrombolysis_prob)

# --- Mortality probability ---
mortality_prob = np.full(N, 0.10)
mortality_prob += (age - 60) * 0.01                          # +1pp per year above 60
mortality_prob[ethnicity == 'asian'] += 0.05
mortality_prob[ethnicity == 'black'] += 0.10
mortality_prob[male == 1] += 0.10

# --- Thrombolysis effect on mortality ---
# Baseline: halves mortality (multiplier 0.5 → reduction of 0.5)
# Each "half as effective" halves the reduction: multiplier 0.75 per condition
reduced_effectiveness = (ethnicity == 'asian') | (ethnicity == 'black')
age_over_80_treated   = (thrombolysis == 1) & (age > 80)
ethnicity_reduced     = (thrombolysis == 1) & reduced_effectiveness
both                  = age_over_80_treated & ethnicity_reduced

thrombolysis_effect = np.where(thrombolysis == 0, 1.0,
    np.where(both, 0.875,
        np.where(age_over_80_treated | ethnicity_reduced, 0.75, 0.5)
    )
)

mortality_prob *= thrombolysis_effect
mortality_prob = np.clip(mortality_prob, 0, 1)

# --- Outcome ---
died = np.random.binomial(1, mortality_prob)

# --- DataFrame ---
df = pd.DataFrame({
    'patient_id':       np.arange(1, N + 1),
    'age':              age,
    'ethnicity':        ethnicity,
    'male':             male,
    'thrombolysis':     thrombolysis,
    'mortality_prob':   np.round(mortality_prob, 4),
    'died':             died
})

# --- Save to CSV ---
df.to_csv('synthetic_stroke_data_1.csv', index=False)